# Three-Way LLM Conversation (OpenAI · Anthropic · Gemini)

This notebook demonstrates a reliable pattern for orchestrating a **three-way conversation between different Large Language Models (LLMs)**—specifically OpenAI (GPT), Anthropic (Claude), and Google Gemini—using **one system prompt and one user prompt per turn**.

## Problem Being Solved
Most LLM APIs are stateless. To simulate a multi-agent conversation, each model must be shown the **entire conversation history** every time it is called.

The challenge is to:
- Keep **distinct personalities** per model
- Maintain **conversation continuity**
- Avoid complex role juggling (`assistant`, `user`, etc.)
- Remain compatible across providers

## Solution Approach
This notebook uses a **round-robin orchestration pattern**:
1. Each model has a **fixed system prompt** defining its persona
2. The **entire conversation so far** is passed as a single user message
3. The model is instructed to generate **only its next line**
4. The response is appended to the shared transcript
5. The process repeats for the next model

This approach is simple, robust, and works consistently across providers.

## Scenario
The models are personified as three stand-up comedians performing together:
- **Oliver (GPT)** — deadpan, analytical
- **Andrew (Claude)** — practical, crowd-working
- **George (Gemini)** — absurd, chaotic

The scenario is illustrative; the same pattern applies to:
- Multi-agent planning
- Debate systems
- Role-based assistants
- Cross-model evaluation

## Key Design Principles
- One system prompt per model
- One user prompt per call
- Full transcript passed every turn
- Explicit speaker labels in the transcript
- Strict instruction to generate a single response

## Requirements
- API keys for:
  - OpenAI
  - Anthropic
  - Google Gemini

Environment variables:
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GOOGLE_API_KEY`

## Notes
- The OpenAI SDK is used with alternative `base_url` values for Anthropic and Gemini via their OpenAI-compatible endpoints.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ


In [3]:
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"


openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url,
)

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5-20251001"
gemini_model = "gemini-3.5-flash-lite"


In [4]:
openai_prompt= """
given this topic: “Should companies replace junior developers with AI coding agents?”
your position is you are a CTO and argues for productivity, lower costs and faster delivery 
"""
anthropic_prompt= f"""
given this topic: “Should companies replace junior developers with AI coding agents?”
your position is you are a senior developer, focuses on code quality, supervision and losing the pipeline of future senior developers
"""
gemini_prompt= f"""
given this topic: “Should companies replace junior developers with AI coding agents?”
your position is you are a junior developer, focuses on employment, learning opportunities, fairness and long-term effects
"""


In [5]:
conversation = [
    ("CTO", "Hi sernior dev and junior dev"),
    ("senior dev", "hello cto and junior dev"),
    ("junior dev", "hello cto and senior dev")
]

In [6]:
def format_conversation(conversation):
    return "\n".join(f"{speaker}: {text}" for speaker, text in conversation )

In [7]:
def next_line(client, model, system_prompt, speaker_name, conversation):
    convo = format_conversation(conversation)

    user_prompt = (
        f"You are {speaker_name} in a three-person debate.\n"
        f"The topic is: Should companies replace junior developers"
        f"with AI coding agents?\n\n"
        "Rules:\n"
        "- Stay in character and defend your assigned perspective.\n"
        "- Responsd directly to the previous arguments when relervant.\n"
        "- Challenge weak assumptions instead of automatically agreeing.\n"
        "- Introduce a new argument or practical consideration.\n"
        "- Do not write the oter participant's consideration \n"
        "- No narration, labels, or stage directions,\n"
        "- Keep the response to 2-4 sentences.\n\n"
        "Converstaion so far:\n"
        f"{convo}\n\n"
        f"Now write {speaker_name}'s next response"
    )

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system", "content": system_prompt},
            {"role":"user", "content": user_prompt}
        ]
    )
    return resp.choices[0].message.content

In [ ]:
for msg in conversation:
    display(Markdown(f"### {msg[0]}:\n{msg[1]}\n"))

for _ in range(5):
    cto_next = next_line(
        openai, gpt_model, openai_prompt, "CTO", conversation
    )
    conversation.append(("CTO", cto_next))
    display(Markdown(f"### CTO:\n{cto_next}\n"))

    senior_dev_next = next_line(
        anthropic, claude_model, anthropic_prompt,
        "Senior Dev", conversation
    )
    conversation.append(("Senior Dev", senior_dev_next))
    display(Markdown(f"### Senior Dev:\n{senior_dev_next}\n"))

    junior_dev_next = next_line(
        gemini, gemini_model, gemini_prompt,
        "Junior Dev", conversation
    )
    conversation.append(("Junior Dev", junior_dev_next))
    display(Markdown(f"### Junior Dev:\n{junior_dev_next}\n"))